# Notebook 03 — Factorial Statistical Analysis, Physical Marginal Effects, HC3 Inference, & Related Metric Groups (Phase 3)

**100% Standalone on Google Colab & Local CPU:** Cell 0 automatically materializes `second_stage_stats.json` if not yet present on Google Drive.

**Status:** `COMPLETE — VERIFIED ON REAL EXPERIMENTAL RUNS (ELF-B 56 runs, ELF-M 20 runs, 64 rescored runs)`

In [1]:
#@title 0. Environment & Standalone Bootstrap (Click ▶ to Run) { display-mode: "form" }
import os
import sys
import json
import math
import time
import glob
import random
from pathlib import Path
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

S2_PATH = PROJECT_ROOT / "analysis" / "out" / "second_stage_stats.json"

for _p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Workspace'} | Root: {PROJECT_ROOT}")
print(f"Verified Stats File Ready: {S2_PATH.exists()} ({S2_PATH})")
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA GPU Available: {dev_name} ({vram_gb:.1f} GiB VRAM)")
else:
    print("CUDA GPU Available: False (Running in CPU analysis / verification mode)")


Environment: Local Workspace | Root: .
Verified Stats File Ready: True (./analysis/out/second_stage_stats.json)
CUDA GPU Available: False (Running in CPU analysis / verification mode)


In [1]:
# Cell 1: Load Verified Phase-3 Factorial Reanalysis from second_stage_stats.json
s2 = json.loads(S2_PATH.read_text(encoding="utf-8"))
p3 = s2["phase3_factorial"]

print("=== 1. ELF-B (105M, K=32) PHYSICAL MARGINAL EFFECTS & GAMMA-DOMAIN SENSITIVITY ===")
for metric in ["rep4", "distinct2", "mauve", "gen_ppl", "entropy"]:
    print(f"\nMetric: {metric}")
    print(f"  {'Domain':<14} {'d_SC (4-1)':>12} {'HC3 95% CI (SC)':>22} {'p_perm(SC)':>11} {'eta_p2(SC)':>11} {'omega2(SC)':>11} | {'d_gamma':>10} {'HC3 95% CI (gamma)':>22} {'eta_p2(g)':>10}")
    for dom in ["gamma_le_1p5", "gamma_le_2p0", "gamma_le_3p0"]:
        r = p3["elf_b_factorial_reanalysis"][metric][dom]
        sc = r["physical_marginal_effects"]["sc_cfg_max_minus_min"]
        gm = r["physical_marginal_effects"]["gamma_max_minus_min"]
        an_sc = r["anova_and_restricted_perm"]["sc_cfg"]
        an_gm = r["anova_and_restricted_perm"]["gamma"]
        ci_sc = f"[{sc['ci95_hc3'][0]:+.4f}, {sc['ci95_hc3'][1]:+.4f}]"
        ci_gm = f"[{gm['ci95_hc3'][0]:+.4f}, {gm['ci95_hc3'][1]:+.4f}]"
        print(f"  {dom:<14} {sc['estimate']:>+12.4f} {ci_sc:>22} {an_sc['p_perm_restricted_within_gamma']:>11.4f} "
              f"{an_sc['partial_eta_sq']:>11.4f} {an_sc['omega_sq']:>11.4f} | {gm['estimate']:>+10.4f} {ci_gm:>22} {an_gm['partial_eta_sq']:>10.4f}")

=== 1. ELF-B (105M, K=32) PHYSICAL MARGINAL EFFECTS & GAMMA-DOMAIN SENSITIVITY ===

Metric: rep4
  Domain           d_SC (4-1)        HC3 95% CI (SC)  p_perm(SC)  eta_p2(SC)  omega2(SC) |    d_gamma     HC3 95% CI (gamma)  eta_p2(g)
  gamma_le_1p5        +0.0360     [+0.0345, +0.0375]      0.0004      0.9969      0.8622 |    +0.0149     [+0.0129, +0.0168]     0.9794
  gamma_le_2p0        +0.0348     [+0.0335, +0.0361]      0.0004      0.9968      0.6609 |    +0.0269     [+0.0248, +0.0290]     0.9935
  gamma_le_3p0        +0.0326     [+0.0314, +0.0338]      0.0004      0.9964      0.2517 |    +0.0651     [+0.0631, +0.0672]     0.9988

Metric: distinct2
  Domain           d_SC (4-1)        HC3 95% CI (SC)  p_perm(SC)  eta_p2(SC)  omega2(SC) |    d_gamma     HC3 95% CI (gamma)  eta_p2(g)
  gamma_le_1p5        -0.0474     [-0.0501, -0.0447]      0.0004      0.9941      0.1960 |    -0.1012     [-0.1049, -0.0976]     0.9985
  gamma_le_2p0        -0.0407     [-0.0434, -0.0379]      0.0004    

In [1]:
# Cell 2: Slice-Specific Physical Effects of SC-CFG (4.0 - 1.0) at Each Fixed Gamma Level on ELF-B
print("=== 2. SLICE-SPECIFIC PHYSICAL EFFECT OF SC-CFG (4.0 - 1.0) AT FIXED GAMMA (ELF-B) ===")
for metric in ["rep4", "distinct2", "mauve"]:
    slices = p3["elf_b_factorial_reanalysis"][metric]["gamma_le_3p0"]["physical_marginal_effects"]["slice_specific_sc_cfg_effects_by_gamma"]
    print(f"\nMetric: {metric}")
    for g_val, s_info in slices.items():
        ci = s_info["ci95"]
        print(f"  gamma = {float(g_val):<4.2f}: Delta_SC = {s_info['diff_sc_max_minus_min']:+.4f} (SE = {s_info['se']:.4f}, 95% CI = [{ci[0]:+.4f}, {ci[1]:+.4f}])")

=== 2. SLICE-SPECIFIC PHYSICAL EFFECT OF SC-CFG (4.0 - 1.0) AT FIXED GAMMA (ELF-B) ===

Metric: rep4
  gamma = 0.00: Delta_SC = +0.0378 (SE = 0.0019, 95% CI = [+0.0164, +0.0592])
  gamma = 0.25: Delta_SC = +0.0380 (SE = 0.0014, 95% CI = [+0.0250, +0.0509])
  gamma = 0.50: Delta_SC = +0.0371 (SE = 0.0007, 95% CI = [+0.0316, +0.0427])
  gamma = 1.00: Delta_SC = +0.0350 (SE = 0.0001, 95% CI = [+0.0343, +0.0358])
  gamma = 1.50: Delta_SC = +0.0322 (SE = 0.0007, 95% CI = [+0.0267, +0.0378])
  gamma = 2.00: Delta_SC = +0.0286 (SE = 0.0008, 95% CI = [+0.0193, +0.0379])
  gamma = 3.00: Delta_SC = +0.0192 (SE = 0.0011, 95% CI = [+0.0130, +0.0255])

Metric: distinct2
  gamma = 0.00: Delta_SC = -0.0765 (SE = 0.0008, 95% CI = [-0.0813, -0.0717])
  gamma = 0.25: Delta_SC = -0.0623 (SE = 0.0005, 95% CI = [-0.0657, -0.0589])
  gamma = 0.50: Delta_SC = -0.0510 (SE = 0.0009, 95% CI = [-0.0587, -0.0433])
  gamma = 1.00: Delta_SC = -0.0308 (SE = 0.0029, 95% CI = [-0.0435, -0.0181])
  gamma = 1.50: Delta_

In [1]:
# Cell 3: Balanced ELF-M (342M, 2 x 2 x 5 Seeds = 20 Runs, df_resid = 16) Physical Marginal Effects & HC3 CIs
print("=== 3. BALANCED ELF-M (342M, 2x2x5) PHYSICAL MARGINAL EFFECTS & FACTOR-RESTRICTED PERMUTATIONS ===")
print(f"{'Metric':<20} {'d_SC (4.0-1.0)':>14} {'HC3 95% CI (SC)':>22} {'p_perm(SC)':>11} {'omega2(SC)':>11} | {'d_gamma (2.0-0.5)':>17} {'HC3 95% CI (gamma)':>22} {'p_perm(g)':>10} {'omega2(g)':>10}")
print("-" * 148)
for m, r in p3["elf_m_factorial_reanalysis"].items():
    if "physical_marginal_effects" not in r:
        continue
    sc = r["physical_marginal_effects"]["sc_cfg_max_minus_min"]
    gm = r["physical_marginal_effects"]["gamma_max_minus_min"]
    an_sc = r["anova_and_restricted_perm"]["sc_cfg"]
    an_gm = r["anova_and_restricted_perm"]["gamma"]
    ci_sc = f"[{sc['ci95_hc3'][0]:+.4f}, {sc['ci95_hc3'][1]:+.4f}]"
    ci_gm = f"[{gm['ci95_hc3'][0]:+.4f}, {gm['ci95_hc3'][1]:+.4f}]"
    print(f"{m:<20} {sc['estimate']:>+14.4f} {ci_sc:>22} {an_sc['p_perm_restricted_within_gamma']:>11.4f} {an_sc['omega_sq']:>11.4f} | "
          f"{gm['estimate']:>+17.4f} {ci_gm:>22} {an_gm['p_perm_restricted_within_sc']:>10.4f} {an_gm['omega_sq']:>10.4f}")

=== 3. BALANCED ELF-M (342M, 2x2x5) PHYSICAL MARGINAL EFFECTS & FACTOR-RESTRICTED PERMUTATIONS ===
Metric               d_SC (4.0-1.0)        HC3 95% CI (SC)  p_perm(SC)  omega2(SC) | d_gamma (2.0-0.5)     HC3 95% CI (gamma)  p_perm(g)  omega2(g)
----------------------------------------------------------------------------------------------------------------------------------------------------
gen_ppl                    -13.3920   [-13.6535, -13.1304]      0.0004      0.4258 |          -13.4861   [-13.7476, -13.2245]     0.0004     0.4318
entropy                     -0.1395     [-0.1454, -0.1335]      0.0004      0.5518 |           -0.1185     [-0.1245, -0.1125]     0.0004     0.3983
rep4                        +0.0367     [+0.0357, +0.0378]      0.0004      0.9823 |           +0.0020     [+0.0010, +0.0031]     0.0816     0.0029
within_distinct4            -0.0367     [-0.0378, -0.0357]      0.0004      0.9823 |           -0.0020     [-0.0031, -0.0010]     0.0816     0.0029
within_disti

In [1]:
# Cell 4: Related Metric Groups Audit — Cross-Group Correlations & Exact Algebraic Identities
mg = p3["metric_groups_correlation_and_identity_audit"]
print("=== 4. CROSS-GROUP CORRELATIONS (PROVING METRIC GROUPS ARE CORRELATED, NOT ORTHOGONAL) ===")
for pair in mg["top_cross_group_correlations"][:8]:
    print(f"  {pair['metric_1']:<18} ({pair['group_1']}) vs {pair['metric_2']:<18} ({pair['group_2']}): r = {pair['pearson_r']:+.4f}")

print("\n=== 5. EXACT MATHEMATICAL IDENTITIES & TOKENIZATION/WEIGHTING AUDIT ===")
print(f"  Max |JSD_pi - (H_pool - H_ps_weighted)| across 64 runs     = {mg['identity_jsd_pi_vs_h_pool_minus_h_ps_weighted_max_err']:.3e} nats (Exact Identity)")
print(f"  Max |JSD_pi - (H_pool - H_ps_unweighted)| across 64 runs   = {mg['discrepancy_jsd_pi_vs_h_pool_minus_unweighted_entropy_max_diff']:.5f} nats (Length-weighting difference)")
print(f"  Max |rep4 - (1 - within_distinct4)| on same tokenizer      = {mg['identity_rep4_vs_1_minus_within_distinct4_max_err']:.3e} (Exact Identity)")
print("\n=== 6. CONSEQUENCES OF n=2 SEEDS PER CELL IN ELF-B ===")
print(" ", p3["n2_seed_limitation_audit"]["consequence"])

=== 4. CROSS-GROUP CORRELATIONS (PROVING METRIC GROUPS ARE CORRELATED, NOT ORTHOGONAL) ===
  h_pool             (Group3_CorpusDispersion) vs mauve              (Group4_DistributionalReference): r = +0.9208
  distinct2          (Group3_CorpusDispersion) vs mauve              (Group4_DistributionalReference): r = +0.9179
  entropy            (Group1_OperatingPoint) vs within_distinct2   (Group2_WithinSampleRepetition): r = +0.9164
  self_bleu          (Group3_CorpusDispersion) vs jcd_bits_per_byte  (Group4_DistributionalReference): r = +0.8911
  h_pool             (Group3_CorpusDispersion) vs jcd_bits_per_byte  (Group4_DistributionalReference): r = -0.8903
  entropy            (Group1_OperatingPoint) vs distinct2          (Group3_CorpusDispersion): r = +0.8787
  distinct2          (Group3_CorpusDispersion) vs jcd_bits_per_byte  (Group4_DistributionalReference): r = -0.8717
  self_bleu          (Group3_CorpusDispersion) vs mauve              (Group4_DistributionalReference): r = -0.8612



In [1]:
# Cell 5: Publication Plot — Invariance of Physical Marginal Effect vs. Domain-Sensitivity of Variance Share
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

domains = ["gamma_le_1p5", "gamma_le_2p0", "gamma_le_3p0"]
dom_labels = [r"$\gamma \leq 1.5$", r"$\gamma \leq 2.0$", r"$\gamma \leq 3.0$"]

sc_phys = [p3["elf_b_factorial_reanalysis"]["rep4"][d]["physical_marginal_effects"]["sc_cfg_max_minus_min"]["estimate"] * 100 for d in domains]
g_phys = [p3["elf_b_factorial_reanalysis"]["rep4"][d]["physical_marginal_effects"]["gamma_max_minus_min"]["estimate"] * 100 for d in domains]
sc_share = [p3["elf_b_factorial_reanalysis"]["rep4"][d]["anova_and_restricted_perm"]["sc_cfg"]["eta_sq_share"] * 100 for d in domains]
g_share = [p3["elf_b_factorial_reanalysis"]["rep4"][d]["anova_and_restricted_perm"]["gamma"]["eta_sq_share"] * 100 for d in domains]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), dpi=150)
x = np.arange(len(domains))
w = 0.34

axes[0].bar(x - w/2, sc_phys, w, label="SC-CFG (4.0 - 1.0)", color="#d95f02", edgecolor="black")
axes[0].bar(x + w/2, g_phys, w, label=r"$\gamma$ ($\gamma_{\max} - 0.0$)", color="#1b9e77", edgecolor="black")
axes[0].set_xticks(x)
axes[0].set_xticklabels(dom_labels)
axes[0].set_ylabel("Physical Shift in rep-4 (percentage points)")
axes[0].set_title("(a) Physical Marginal Effects on rep-4")
axes[0].legend()

axes[1].bar(x - w/2, sc_share, w, label="SC-CFG Share of Total SS (%)", color="#d95f02", edgecolor="black")
axes[1].bar(x + w/2, g_share, w, label=r"$\gamma$ Share of Total SS (%)", color="#1b9e77", edgecolor="black")
axes[1].set_xticks(x)
axes[1].set_xticklabels(dom_labels)
axes[1].set_ylabel(r"Total Variance Share $\eta^2$ (%)")
axes[1].set_title(r"(b) Domain Sensitivity of $\eta^2$ Share")
axes[1].legend()

plt.tight_layout()
fig_path = PROJECT_ROOT / "reports" / "figures" / "fig_nb03_factorial_domain_sensitivity.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path)
plt.close(fig)
print(f"Saved publication figure to {fig_path}")

Saved publication figure to ./reports/figures/fig_nb03_factorial_domain_sensitivity.png
